In [ ]:
DATA_DIR = "./.napistu_data"
STORE_DIR = "./.store"
OVERWRITE_STORE_DIR = False
NAPISTU_DATA_INSTANCE_NAME = "edge_prediction"
USE_EDGE_ENCODER = True

# download the model from GCS and save it to a local directory to initialize a NapistuDataStore
# or load an existing store
napistu_data_store = gcs_model_to_store(
    napistu_data_dir = DATA_DIR,
    store_dir = STORE_DIR,
    overwrite_store_dir = OVERWRITE_STORE_DIR,
)

# define the experiment config (could also be done in a config file)
config = ExperimentConfig(
    model = ModelConfig(
        encoder = ENCODERS.GRAPH_CONV,
        head = HEADS.DOT_PRODUCT,
        use_edge_encoder = USE_EDGE_ENCODER,
    ),
    data = DataConfig(
        store_dir = STORE_DIR,
        sbml_dfs_path = napistu_data_store.sbml_dfs_path,
        napistu_graph_path = napistu_data_store.napistu_graph_path,
        napistu_data_name = NAPISTU_DATA_INSTANCE_NAME,
    ),
    task = TaskConfig(
        edge_prediction_neg_sampling_stratify_by = "edge_strata_by_node_type",
        task = "edge_prediction",
    ),
    training = TrainingConfig(
        scheduler = "plateau",
        early_stopping_patience = 20,
        save_checkpoints=True,
        batches_per_epoch = 10,
        #accelerator="cpu"
    )
)

from napistu_torch.lightning.workflows import prepare_experiment, resume_experiment, log_experiment_overview

experiment_dict = prepare_experiment(config)
log_experiment_overview(experiment_dict)